In [ ]:
# -*- coding: utf-8 -*-
"""
Generate anonymized GI (pathology & endoscopy) reports according to the latest clinical guidelines.
Features:
- 8 patient categories based on risk stratification.
- 39 extracted fields matching EU Guidelines 2025 schema.
- Mandatory report components from Appendix A & B.
- All patients require precancerous staging (endoscopic and pathological).
- Role-separated content (endoscopy never includes histology-only items).
- Random visible/nonvisible scenarios for LGD/HGD.
- Error injection (missing/wrong) with separate explanation files.
- Debug mode for gradual testing (1 pair → 1 full categories → full batch).
- DeepSeek API compatible.
"""

import os
import re
import json
import random
import time
import zipfile
import datetime  
from typing import Dict, Any, List, Optional, Tuple
from openai import OpenAI

GENERATE_WRONG = False
# =======================
# API Key & Model (DeepSeek)
# =======================
API_KEY = os.getenv("SILICONFLOW_API_KEY", "")
BASE_URL = "https://api.siliconflow.cn/v1"
MODEL_NAME = "deepseek-ai/DeepSeek-V4-Flash"   

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

# =======================
# Token Limits
# =======================
MAX_REPORT_TOKENS = 4096  
MAX_EXPLANATION_TOKENS = 500
MAX_EXTRACTION_TOKENS = 800

# =======================
# Output options
# =======================
REPORTS_DIR = "reports"
ZIP_FILENAME = "all_reports.zip"
MAKE_MASTER_PER_PATIENT = True
CLEAN_AFTER_ZIP = False
# =======================
# 小批量测试配置（每次只测一个类别，生成全部内容）
# =======================
BATCH_TEST_CATEGORY = "Post-Endoscopic Resection"
BATCH_TEST_NUM_PAIRS = 6                         

# =======================
# Categories (基于风险分层，不含指南名称)
# =======================
categories = [
    "Healthy State",
    "Atrophic Gastritis",
    "Gastric Intestinal Metaplasia",
    "Low-grade Dysplasia",
    "High-grade Dysplasia",
    "Early Gastric Carcinoma",
    "Post-Endoscopic Resection",
    "Special Populations",
]

CATEGORY_CODES = {
    "Healthy State": "HS",
    "Atrophic Gastritis": "AG",
    "Gastric Intestinal Metaplasia": "GIM",
    "Low-grade Dysplasia": "LGD",
    "High-grade Dysplasia": "HGD",
    "Early Gastric Carcinoma": "EGC",
    "Post-Endoscopic Resection": "PER",
    "Special Populations": "SP",
}

# =======================
# DEBUG / PRODUCTION 模式
# =======================
DEBUG_MODE = True                  # True=调试模式, False=批量生成
DEBUG_CATEGORIES = ["Post-Endoscopic Resection", "Early Gastric Carcinoma"]  # 调试时只跑这些类别
DEBUG_NUM_PAIRS = 1                # 调试时每个类别生成几对（正确+错误）
PRODUCTION_NUM_PAIRS = 6          # 批量生产时每类生成的数量

# =======================
# Demographic Pools
# =======================
eu_nationalities = [
    'Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czechia',
    'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary',
    'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta',
    'Netherlands', 'Poland', 'Portugal', 'Romania', 'Slovakia', 'Slovenia',
    'Spain', 'Sweden'
]
allowed_nationalities = eu_nationalities + [
    "United States of America",
    "United Kingdom of Great Britain and Northern Ireland",
]

genders = ['Male', 'Female']
races = ['White', 'Black', 'Asian', 'Hispanic', 'Mixed', 'Other']
lifestyle_options = [
    "Non-smoker",
    "Smoker",
]

# 家族史仅限"一级亲属胃癌"，不包含其他胃肠道疾病
family_histories = [
    'No family history',
    'First-degree family history of gastric cancer',
]

# =======================
# EXTRACTION SCHEMA (39 fields, EU Guidelines 2025)
# =======================
EXTRACTION_KEYS = [
    "kimura_takemoto_classification",
    "eggim_score",
    "olga_stage",
    "olgim_stage",
    "helicobacter_pylori_status",
    "atrophic_gastritis",
    "incomplete_intestinal_metaplasia",
    "family_history",
    "dysplasia",
    "dysplasia_grade",
    "dysplasia_visibility",
    "indefinite_for_dysplasia",
    "neoplastic_lesion_visibility",
    "neoplastic_lesion",
    "lesion_paris_classification",
    "neoplastic_lesion_size",
    "neoplastic_lesion_ulceration",
    "differentiation_status",
    "pt_category",
    "submucosal_invasion",
    "submucosal_invasion_depth",
    "lymphovascular_invasion",
    "lymph_node_metastasis_risk",
    "recommended_treatment",
    "en_bloc_resection",
    "piecemeal_resection",
    "en_bloc_resection_margins",
    "horizontal_margin_status",
    "vertical_margin_status",
    "resection_curability",
    "autoimmune_gastritis",
    "common_variable_immunodeficiency",
    "gastric_malt_lymphoma",
    "hereditary_syndrome",
    "pepsinogen_i_level",
    "pepsinogen_ii_level",
    "patient_age",
    "life_expectancy",
    "smoking_status"
]

ALLOWED: Dict[str, set] = {
    "kimura_takemoto_classification": {"o3", "o2", "o1", "c3", "c2", "c1", "c0", "not_mentioned"},
    "olga_stage": {"0", "i", "ii", "iii", "iv", "not_mentioned"},
    "olgim_stage": {"0", "i", "ii", "iii", "iv", "not_mentioned"},
    "helicobacter_pylori_status": {"positive", "negative", "persistent", "not_mentioned"},
    "atrophic_gastritis": {"yes", "no", "not_mentioned"},
    "incomplete_intestinal_metaplasia": {"yes", "no", "not_mentioned"},
    "family_history": {"yes", "no", "not_mentioned"},
    "dysplasia": {"yes", "no", "highly_likely", "not_mentioned"},
    "dysplasia_grade": {"high_grade", "low_grade", "not_mentioned"},
    "dysplasia_visibility": {"visible", "nonvisible", "not_mentioned"},
    "indefinite_for_dysplasia": {"yes", "no", "not_mentioned"},
    "neoplastic_lesion_visibility": {"visible", "nonvisible", "not_mentioned"},
    "neoplastic_lesion": {"yes", "no", "highly_likely", "not_mentioned"},
    "lesion_paris_classification": {"0_iia", "other", "not_mentioned"},
    "neoplastic_lesion_ulceration": {"yes", "no", "not_mentioned"},
    "differentiation_status": {"differentiated", "undifferentiated", "not_mentioned"},
    "pt_category": {"pt1a", "pt1b", "other", "not_mentioned"},
    "submucosal_invasion": {"yes", "no", "not_mentioned"},
    "submucosal_invasion_depth": {"le_500um", "gt_500um", "not_mentioned"},
    "lymphovascular_invasion": {"yes", "no", "not_mentioned"},
    "lymph_node_metastasis_risk": {"lt_0.5_percent", "0.5_to_1_percent", "lt_3_percent", "ge_3_percent", "not_mentioned"},
    "recommended_treatment": {"surgical_treatment", "conservative_management", "endoscopic_resection", "endoscopic_submucosal_dissection", "endoscopic_mucosal_resection", "not_mentioned"},
    "en_bloc_resection": {"yes", "no", "not_mentioned"},
    "piecemeal_resection": {"yes", "no", "not_mentioned"},
    "en_bloc_resection_margins": {"r0", "rx", "r1", "not_mentioned"},
    "horizontal_margin_status": {"positive", "negative", "not_mentioned"},
    "vertical_margin_status": {"positive", "negative", "not_mentioned"},
    "resection_curability": {"curative", "noncurative", "not_mentioned"},
    "autoimmune_gastritis": {"yes", "no", "not_mentioned"},
    "common_variable_immunodeficiency": {"yes", "no", "not_mentioned"},
    "gastric_malt_lymphoma": {"active", "in_remission", "not_mentioned"},
    "hereditary_syndrome": {"yes", "no", "not_mentioned"},
    "life_expectancy": {"lt_10_years", "ge_10_years", "not_mentioned"},
    "smoking_status": {"smoker", "non_smoker", "not_mentioned"},
}

NUMERIC_STRING_KEYS = {
    "eggim_score": r"^(not_mentioned|10|[0-9])$",
    "neoplastic_lesion_size": r"^(not_mentioned|(?:\d+(?:\.\d+)?|\.\d+)\s*mm)$",
    "pepsinogen_i_level": r"^(not_mentioned|\d+(?:\.\d+)?\s*ng/ml)$",
    "pepsinogen_ii_level": r"^(not_mentioned|\d+(?:\.\d+)?\s*ng/ml)$",
    "patient_age": r"^(not_mentioned|\d+)$",
}

# =======================
# Load MAPS III Appendix Requirements from Excel (neutral terms)
# =======================

# Default hardcoded requirements (no guideline names)
DEFAULT_ENDOSCOPY_BASE = [
    "Paris classification of any visible lesion. If no lesion, state 'No visible lesion'.",
    "Ulceration status (Yes/No). If no lesion, state 'No ulceration'.",
    "Lesion size in mm. If no lesion, state 'Not applicable'.",
    "Stage of precancerous conditions (MANDATORY for ALL patients): "
    "Explicitly state Kimura–Takemoto classification and the EGGIM score (0-10) based on virtual chromoendoscopy findings. ",
]

DEFAULT_ENDOSCOPY_ESD = [
    "Exact location of the lesion/resection site.",
    "En bloc vs. piecemeal resection (if resection was performed).",
]

DEFAULT_PATHOLOGY_BASE = [
    "Chronic gastritis: Yes/No and severity (mild/moderate/severe).",
    "Activity: Yes/No and severity (mild/moderate/severe).",
    "Glandular atrophy: none/mild/moderate/severe.",
    "Intestinal metaplasia: none/mild/moderate/severe; if present, specify complete vs. incomplete. Do not use type I/II/III subtypes.",
    "Dysplasia: no/low-grade/high-grade/indefinite (if applicable). If indefinite, explicitly state 'indefinite for dysplasia'.",
    "H. pylori status: Yes/No and detection method (e.g., Giemsa, IHC).",
]

DEFAULT_PATHOLOGY_ESD = [
    "Most severe histology and differentiation.",
    "Specimen size in mm.",
    "Horizontal margin status: negative (HM0) or positive (HM1).",
    "Vertical margin status: negative (VM0) or positive (VM1).",
    "Maximum depth of submucosal invasion in µm.",
    "Lymphatic and venous infiltration: L0/L1 and V0/V1.",
    "Overall resection status: R0, RX, R1.",
    "pT category (pT1a, pT1b) if applicable for carcinoma.",
    "Lymph node metastasis risk: explicitly state the risk category (e.g., <0.5%, 0.5-1%, <3%, ≥3%) based on histology and depth of invasion.",
]

# Categories that require ESD items
CATEGORIES_REQUIRING_ESD = {
    "Low-grade Dysplasia",
    "High-grade Dysplasia",
    "Early Gastric Carcinoma",
    "Post-Endoscopic Resection",
}

def get_endoscopy_requirements(category: str) -> List[str]:
    # 所有类别的内镜报告都只使用 BASE 项（Paris, 溃疡, 大小, 癌前分期）
    return DEFAULT_ENDOSCOPY_BASE.copy()

def get_pathology_requirements(category: str) -> List[str]:
    reqs = DEFAULT_PATHOLOGY_BASE.copy()
    if category == "Post-Endoscopic Resection":   # 仅PER有切除后信息
        reqs.extend(DEFAULT_PATHOLOGY_ESD)
    return reqs


# =======================
# Clinical Facts Generators (Pre-calculation)
# =======================

def calculate_eggim(im_antrum: str, im_corpus: str) -> Dict[str, Any]:
    """
    Calculate EGGIM scores (0-2 per site, total 0-10).
    Standard five sites: antrum (two sites), incisura, corpus (two sites).
    """
    map_score = {"none": 0, "mild": 1, "moderate": 2, "severe": 2}
    antrum_score = map_score.get(im_antrum, 0)
    corpus_score = map_score.get(im_corpus, 0)

    # 切迹评分在窦和体之间取合理值
    incisura = random.randint(max(0, min(antrum_score, corpus_score)-1),
                              min(2, max(antrum_score, corpus_score)+1))
    # 体小弯/大弯可略有差异
    body_small = random.randint(max(0, corpus_score-1), min(2, corpus_score+1))
    body_large = random.randint(max(0, corpus_score-1), min(2, corpus_score+1))
    total = antrum_score * 2 + incisura + body_small + body_large
    return {
        "eggim_antrum": antrum_score,
        "eggim_incisura": incisura,
        "eggim_body_small": body_small,
        "eggim_body_large": body_large,
        "eggim_total": min(total, 10)
    }

def calculate_stages_and_kt(atrophy_antrum: str, atrophy_corpus: str,
                            im_antrum: str, im_corpus: str) -> Dict[str, Any]:
    """
    根据胃窦、胃体的萎缩/肠化程度（0-3分），通过二维矩阵确定OLGA/OLGIM分期。
    MAPS III 推荐使用矩阵而非总分法。
    """
    severity_score = {"none": 0, "mild": 1, "moderate": 2, "severe": 3}
    ant_atrophy = severity_score[atrophy_antrum]
    cor_atrophy = severity_score[atrophy_corpus]
    ant_im = severity_score[im_antrum]
    cor_im = severity_score[im_corpus]

    # OLGA/OLGIM 分期矩阵 (行 = 胃窦, 列 = 胃体)
    staging_matrix = [
        # corpus 0   1   2   3
        ["0",  "I", "II","II"],   # antrum 0
        ["I",  "I", "II","III"],  # antrum 1
        ["II", "II","III","IV"],  # antrum 2
        ["III","III","IV","IV"]   # antrum 3
    ]
    
    olga_stage = staging_matrix[ant_atrophy][cor_atrophy]
    olgim_stage = staging_matrix[ant_im][cor_im]

    # Kimura-Takemoto 基于萎缩分布（概率加权，更贴近临床）
    severity_score = {"none": 0, "mild": 1, "moderate": 2, "severe": 3}
    ant_score = severity_score[atrophy_antrum]
    cor_score = severity_score[atrophy_corpus]

    if ant_score == 0 and cor_score == 0:
        kt = "c0"
    elif ant_score > 0 and cor_score == 0:
        # 仅胃窦萎缩 → C-1 或 C-2，权重偏向 C-1
        kt = random.choice(["c1", "c1", "c2"])
    elif cor_score == 1:
        # 胃体轻度萎缩 → 大概率 C-2/C-3，极少数可报 O-1
        kt = random.choice(["c2", "c2", "c3", "o1"])
    elif cor_score == 2:
        # 胃体中重度萎缩 → 可能 C-3 或 O-1/O-2
        kt = random.choice(["c3", "o1", "o1", "o2"])
    else:  # cor_score == 3
        # 胃体重度萎缩 → 几乎都是 O 型
        kt = random.choice(["o1", "o2", "o2", "o3"])

    return {
        "olga_stage": olga_stage,
        "olgim_stage": olgim_stage,
        "kimura_takemoto": kt,
        # 可保留原始分数供参考
        "antrum_atrophy_score": ant_atrophy,
        "corpus_atrophy_score": cor_atrophy,
        "antrum_im_score": ant_im,
        "corpus_im_score": cor_im
    }

def generate_esd_features(category: str, force_en_bloc: bool = False) -> Dict[str, Any]:
    """
    生成 ESD 标本的病理特征，严格遵循 MAPS III 附录B 的逻辑。
    """
    # 1. pT category (保持不变)
    if category == "Early Gastric Carcinoma":
        pt = random.choices(["pt1a", "pt1b"], weights=[0.6, 0.4])[0]
    else:
        pt = random.choices(["pt1a", "pt1b"], weights=[0.5, 0.5])[0]

    # 2. Submucosal invasion & En bloc (en bloc 设为 True，代表整块切除)
    if pt == "pt1a":
        sm_invasion = "no"
        sm_depth = "not_mentioned"
    else:
        sm_invasion = "yes"
        sm_depth = random.choices(["le_500um", "gt_500um"], weights=[0.4, 0.6])[0]
    
    # 3. L/V (<-- [修正] 下调概率)
    if sm_depth == "gt_500um":
        l = random.choices(["0", "1"], weights=[0.8, 0.2])[0]  
        v = random.choices(["0", "1"], weights=[0.9, 0.1])[0]  
    else:
        l = random.choices(["0", "1"], weights=[0.95, 0.05])[0]
        v = random.choices(["0", "1"], weights=[0.98, 0.02])[0] 

    if force_en_bloc:
        en_bloc = "yes"
    else:
        en_bloc = random.choices(["yes", "no"], weights=[0.7, 0.3])[0]
    
    # 4. Margins (<-- [修正] HM 概率独立，VM 保持与深度相关)
    hm = random.choices(["negative", "positive"], weights=[0.85, 0.15])[0]   # HM 独立随机
    
    if sm_depth == "gt_500um":
        vm = random.choices(["negative", "positive"], weights=[0.6, 0.4])[0]  # 不变
    else:
        vm = random.choices(["negative", "positive"], weights=[0.85, 0.15])[0] # 不变

    # 5. R status & Curability (<-- [修正] 严格遵循 MAPS III 附录B)
    if vm == "positive":
        r_status = "R1"
        curability = "noncurative"
    elif hm == "positive" and vm == "negative":
        r_status = "RX"
        curability = "noncurative"
    else:  # hm == "negative" and vm == "negative"
        if en_bloc == "yes":
            r_status = "R0"
            curability = "curative"
        else:
            r_status = "RX"   # 分块切除不能算 R0
            curability = "noncurative"

    # 6. LNM risk (<-- [修正] 仅当非R0或pT1b时计算eCura)
    if r_status == "R0" and pt == "pt1a":
        l_nm_risk = "lt_0.5_percent"
    else:
        # 执行 eCura-like 评分 (适用于 R1/RX 或 pT1b)
        score = 0
        if l == "1" or v == "1":
            score += 3
        if vm == "positive":
            score += 1
        if sm_depth == "gt_500um":
            score += 1
        if random.random() < 0.3:
            score += 1  # size >30mm
        if random.random() < 0.2:
            score += 1  # undifferentiated

        if score <= 1:
            l_nm_risk = random.choices(["lt_0.5_percent", "0.5_to_1_percent"], weights=[0.8, 0.2])[0]
        elif score <= 3:
            l_nm_risk = random.choices(["0.5_to_1_percent", "lt_3_percent"], weights=[0.4, 0.6])[0]
        else:
            l_nm_risk = random.choices(["lt_3_percent", "ge_3_percent"], weights=[0.3, 0.7])[0]

    # 7. 返回结果
    return {
        "pt_category": pt,
        "submucosal_invasion": sm_invasion,
        "submucosal_invasion_depth": sm_depth,
        "lymphovascular_invasion": f"L{l}, V{v}",
        "horizontal_margin_status": hm,          # 统一为 "negative"/"positive"
        "vertical_margin_status": vm,            # 统一为 "negative"/"positive"
        "en_bloc_resection_margins": r_status,   # R0 / RX / R1
        "resection_curability": curability,      # "curative" / "noncurative"
        "lymph_node_metastasis_risk": l_nm_risk,
        "en_bloc": en_bloc                       # 记录是否整块
    }

# =======================
# Core Report Generation
# =======================

def generate_report(
    report_type: str,
    patient_name: str,
    category: str,
    gender: str,
    nationality: str,
    age: int,
    family_history: str,
    race: str,
    lifestyle: str,
    pg_i: Optional[float] = None,
    pg_ii: Optional[float] = None,
    clinical_facts: Optional[Dict] = None,
    simulate_error: bool = False,
    error_type: Optional[str] = None,
) -> str:
    if category.lower() == "healthy state":
        condition_text = "a healthy state with no significant pathological findings"
    else:
        condition_text = category

    max_words = int(MAX_REPORT_TOKENS * 0.75)

    prompt = (
        f"Generate a complete, realistic, and anonymized {report_type} report for a patient with {condition_text}.\n"
        f"Patient: '{patient_name}', Gender: {gender}, Nationality: {nationality}, Age: {age}\n"
        f"Race: {race}\n"
        f"Date: use a realistic recent date.\n"
        f"Use standard medical terminology.\n"
    )

    # Special Populations scenario
    if category == "Special Populations":
        if clinical_facts and clinical_facts.get("sp_scenario"):
            sp_scenario = clinical_facts["sp_scenario"]
        else:
            sp_scenario = random.choice([
                "autoimmune_gastritis",
                "common_variable_immunodeficiency",
                "gastric_malt_lymphoma_in_remission",
                "hereditary_syndrome"
            ])
        
        sp_prompt_map = {
            "autoimmune_gastritis": "This patient has autoimmune gastritis (AIG). The report MUST explicitly state the diagnosis of AIG, mention anti-parietal cell antibody status if available, and note the required endoscopic surveillance for gastric cancer and neuroendocrine tumors.",
            "common_variable_immunodeficiency": "This patient has Common Variable Immunodeficiency (CVID). The report MUST explicitly state the CVID diagnosis, mention that a high-quality endoscopy was performed at diagnosis.",
            "gastric_malt_lymphoma_in_remission": "This patient has gastric MALT lymphoma in remission. The report MUST explicitly state the MALT lymphoma history.",
            "hereditary_syndrome": "This patient has a hereditary syndrome conferring gastric cancer risk."
        }
        
        prompt += f"\nSPECIAL POPULATION SCENARIO: {sp_prompt_map[sp_scenario]}\n"
        
        # 如果是遗传性综合征且已存储具体类型，强制报告写出该具体综合征
        if sp_scenario == "hereditary_syndrome" and clinical_facts and clinical_facts.get("specific_syndrome"):
            prompt += f"IMPORTANT: The specific hereditary syndrome is {clinical_facts['specific_syndrome']}. The report MUST explicitly state this exact syndrome.\n"
        
    if age > 80:
        prompt += (
            "\nAGE/LIFE EXPECTANCY NOTE: The patient is over 80 years old. Consider whether intensive surveillance is appropriate, "
            "and mention that clinical judgment regarding life expectancy (>10 years vs. <10 years) should guide the decision to "
            "continue or discontinue screening.\n"
        )

    # 背景描述（共享事实）
    background_prompt = ""
    if clinical_facts:
        bg_parts = []
        if clinical_facts.get("atrophy_antrum") and clinical_facts["atrophy_antrum"] != "none":
            bg_parts.append(f"antrum atrophy: {clinical_facts['atrophy_antrum']}")
        if clinical_facts.get("atrophy_corpus") and clinical_facts["atrophy_corpus"] != "none":
            bg_parts.append(f"corpus atrophy: {clinical_facts['atrophy_corpus']}")
        if clinical_facts.get("im_antrum") and clinical_facts["im_antrum"] != "none":
            bg_parts.append(f"antrum intestinal metaplasia: {clinical_facts['im_antrum']}")
        if clinical_facts.get("im_corpus") and clinical_facts["im_corpus"] != "none":
            bg_parts.append(f"corpus intestinal metaplasia: {clinical_facts['im_corpus']}")
        if clinical_facts.get("lesion_location"):
            bg_parts.append(f"lesion location: {clinical_facts['lesion_location']}") 
        if bg_parts:
            background_prompt = "BACKGROUND (based on clinical data): " + ", ".join(bg_parts) + ".\n"
        if clinical_facts.get("hp_status"):
            background_prompt += f"H. pylori status: {clinical_facts['hp_status']}.\n"
    
    if report_type.lower() == "endoscopy":
        prompt += f"Lifestyle: {lifestyle}\n"
        prompt += f"You MUST explicitly state in the report: 'Family history: {family_history}'.\n"
        if pg_i is not None and pg_ii is not None:
            prompt += f"Serum pepsinogen I: {pg_i} ng/mL, pepsinogen II: {pg_ii} ng/mL.\n"

        if background_prompt:
            prompt += background_prompt

        # 注入预计算的内镜指标（K-T, EGGIM）
        if clinical_facts and not simulate_error:
            kt_value = clinical_facts.get("kimura_takemoto")
            if kt_value:
                # 根据 kt_value 生成正确的文字描述
                kt_desc_map = {
                    "c0": "C-0 (no atrophy)",
                    "c1": "C-1 (atrophy confined to antrum)",
                    "c2": "C-2 (atrophy extends to corpus lesser curvature)",
                    "c3": "C-3 (atrophy extends to corpus greater curvature)",
                    "o1": "O-1 (open type, mild)",
                    "o2": "O-2 (open type, moderate)",
                    "o3": "O-3 (open type, severe)"
                }
                prompt += f"Kimura-Takemoto classification: {kt_desc_map.get(kt_value, kt_value)} (MUST be exactly as stated).\n"
            if clinical_facts.get("eggim_total") is not None:
                prompt += (
                    f"EGGIM scores: antrum {clinical_facts['eggim_antrum']} (each), "
                    f"incisura {clinical_facts['eggim_incisura']}, "
                    f"body small {clinical_facts['eggim_body_small']}, "
                    f"body large {clinical_facts['eggim_body_large']}, "
                    f"total {clinical_facts['eggim_total']}(MUST be exactly as stated).\n"
                )

        # ----- 新增：PER 术前说明 -----
        if category == "Post-Endoscopic Resection":
            prompt += (
                "\nThis is a PREOPERATIVE endoscopy report describing the lesion before endoscopic resection. "
                "Describe the lesion's characteristics (Paris classification, size, ulceration) as if it were still present.\n"
            )

        reqs = get_endoscopy_requirements(category)
        prompt += "\n=== MANDATORY ENDOSCOPY REPORT COMPONENTS (MUST INCLUDE ALL) ===\n"
        for idx, req in enumerate(reqs, 1):
            prompt += f"{idx}. {req}\n"

        prompt += (
            "\nTECHNICAL STANDARD FOR THIS ENDOSCOPY: "
            "The examination must be performed using high quality endoscopy including virtual chromoendoscopy(VCE),"
            "Describe the VCE findings specifically: mucosal and vascular patterns, demarcation lines, "
            "and how VCE aided in staging precancerous conditions (Kimura-Takemoto and EGGIM). "
            "This is mandatory for accurate staging of precancerous conditions.\n"
        )
        prompt += (
            "\nIMPORTANT: Do NOT include histological details such as OLGA/OLGIM stage, "
            "incomplete intestinal metaplasia subtype, or H. pylori detection in this endoscopy report. "
            "Those belong to pathology reports.\n"
        )

        
        if category == "Early Gastric Carcinoma":
            if clinical_facts.get("is_surgery"):
                prompt += "This lesion has high-risk features. Surgical gastrectomy should be strongly considered.\n"
            elif age > 80 or random.random() < 0.1:
                prompt += "Given fragile patient with short life expectancy, conservative management or MDT discussion may be considered.\n"
            else:
                # 只有既非手术也非保守时，才推荐内镜切除
                proc = clinical_facts.get("procedure_type")
                if proc:
                    prompt += f"For this visible neoplastic lesion, {proc} is recommended.\n"

                # LGD/HGD: concise visibility statement for endoscopy
        if category in ["Low-grade Dysplasia", "High-grade Dysplasia"]:
            visibility = clinical_facts.get("is_visible") if clinical_facts else None
            if visibility == "visible":
                prompt += "Endoscopic finding: Visible lesion present. Describe Paris classification, size, ulceration, and mucosal patterns.\n"
            elif visibility == "nonvisible":
                prompt += "Endoscopic finding: No visible lesion. No lesion morphology to describe.\n"
            # else: if visibility is None, do nothing (fallback)

    else:  # pathology
        if background_prompt:
            prompt += background_prompt
        
        # 注入预计算的 OLGA/OLGIM
        if clinical_facts and clinical_facts.get("olga_stage") and not simulate_error:
            prompt += f"OLGA stage: {clinical_facts['olga_stage']}, OLGIM stage: {clinical_facts['olgim_stage']}(MUST be exactly as stated here).\n"

        specimen_types = clinical_facts.get("specimen_types", []) if clinical_facts else ["biopsy"]
        if not specimen_types:
            prompt += "No specimen received. No pathological findings.\n"
        else:
            prompt += "PATHOLOGY REPORT\n"
            if "biopsy" in specimen_types:
                olga = clinical_facts.get("olga_stage", "not_mentioned")
                olgim = clinical_facts.get("olgim_stage", "not_mentioned")
                atro_antrum = clinical_facts.get("atrophy_antrum", "none")
                atro_corpus = clinical_facts.get("atrophy_corpus", "none")
                im_antrum = clinical_facts.get("im_antrum", "none")
                im_corpus = clinical_facts.get("im_corpus", "none")
                hp = clinical_facts.get("hp_status", "not_mentioned")
    
                 # 基础活检描述：标准4块（MAPS III 最低要求）
                biopsy_desc = (
                  "Type: Standard biopsies: antrum x2 (lesser and greater curvature), corpus x2 (lesser and greater curvature).\n"
                )
                # 若存在靶向活检，追加描述
                target_location = clinical_facts.get("lesion_location", "")
                if target_location:
                    biopsy_desc += f"Additionally, targeted biopsies were obtained from the {target_location} for lesion characterization.\n"
    
                biopsy_desc += (
                    "Findings: Based on the background mucosa,\n"
                    f"  - Atrophy: antrum {atro_antrum}, corpus {atro_corpus}\n"
                    f"  - Intestinal metaplasia: antrum {im_antrum}, corpus {im_corpus}\n"
                    f"  - OLGA stage: {olga}\n"
                    f"  - OLGIM stage: {olgim}\n"
                    f"  - H. pylori: {hp}\n"
                )
                if category == "Early Gastric Carcinoma" and clinical_facts.get("is_visible") == "visible":
                    biopsy_desc += "  - Carcinoma: Positive for adenocarcinoma (biopsy sample).\n"
                    if clinical_facts.get("differentiation_status"):
                        biopsy_desc += f"  - Differentiation: {clinical_facts['differentiation_status']}.\n"
                elif category in ["Low-grade Dysplasia", "High-grade Dysplasia"] and clinical_facts.get("is_visible") == "visible":
                    dysplasia_type = clinical_facts.get("dysplasia_type", category.lower())
                    if dysplasia_type == "indefinite":
                        biopsy_desc += "  - Dysplasia: indefinite for dysplasia (biopsy sample).\n"
                    else:
                        biopsy_desc += f"  - Dysplasia: {dysplasia_type} (biopsy sample).\n"
                prompt += biopsy_desc

            if "esd" in specimen_types:
                proc_type = clinical_facts.get("procedure_type", "ESD")
                force_en_bloc = (proc_type == "EMR")  # EMR 必须整块
                esd_features = generate_esd_features(category, force_en_bloc=force_en_bloc)

                # 将 en_bloc 状态传回 clinical_facts，供内镜使用
                if clinical_facts is not None:
                    clinical_facts["en_bloc"] = esd_features["en_bloc"]

                # 动态生成标本类型描述
                if proc_type == "EMR":
                    esd_desc = "--- EMR Resection ---\n"
                    esd_desc += "Type: En bloc EMR specimen (for 0-IIa ≤10mm lesion).\n"
                else:  # ESD
                    esd_desc = "--- ESD Resection ---\n"
                    if esd_features["en_bloc"] == "yes":
                        esd_desc += "Type: En bloc ESD specimen.\n"
                    else:
                        esd_desc += "Type: Piecemeal ESD specimen.\n"

                # 其余特征（不变）
                esd_desc += f"pT Category: {esd_features['pt_category']}\n"
                # 位置
                lesion_loc = clinical_facts.get("lesion_location", "")
                if lesion_loc:
                    esd_desc += f"Lesion location: {lesion_loc}.\n"

                #对所有切除类型都适用
                esd_desc += f"pT Category: {esd_features['pt_category']}\n"
                esd_desc += f"Submucosal invasion: {esd_features['submucosal_invasion']}\n"

                if esd_features['submucosal_invasion_depth'] != "not_mentioned":
                    esd_desc += f"Depth: {esd_features['submucosal_invasion_depth']}\n"
                esd_desc += (
                    f"L/V: {esd_features['lymphovascular_invasion']}\n"
                    f"Margins: horizontal {esd_features['horizontal_margin_status']}, vertical {esd_features['vertical_margin_status']}\n"
                    f"Curability: {esd_features['en_bloc_resection_margins']} resection\n"
                    f"Lymph node metastasis risk: {esd_features['lymph_node_metastasis_risk']}\n"
                )
                prompt += esd_desc

        # 强制项列表，过滤 ESD 特定项
        reqs = get_pathology_requirements(category)
        prompt += "\n=== MANDATORY PATHOLOGY REPORT COMPONENTS ===\n"
        esd_specific_items = ["Horizontal margin status", "Vertical margin status",
                              "Maximum depth of submucosal invasion", "Overall resection status",
                              "pT category", "Lymph node metastasis risk"]
        idx_counter = 1
        for req in reqs:
            if any(key in req for key in esd_specific_items) and "esd" not in specimen_types:
                continue
            prompt += f"{idx_counter}. {req}\n"
            idx_counter += 1

        if "biopsy" in specimen_types:
            prompt += (
                "\nCRITICAL STAGING REQUIREMENT: Based on the severity and distribution of atrophy "
                "and intestinal metaplasia, you MUST explicitly calculate and report "
                "the OLGA stage (0, I, II, III, IV) and OLGIM stage (0, I, II, III, IV) in the report.\n "
            )
            prompt += (
                "\nIMPORTANT: Do NOT include endoscopic classification systems (e.g., Kimura-Takemoto, EGGIM) "
                "in this pathology report. Those belong to endoscopy reports.\n"
            )


    prompt += (
        "\n Clinical findings and staging must align with realistic clinical logic. Keep the report concise. "
        "Do not include redundant information e.g.instrument details, endoscopist names, hospital names, medical record numbers, or specimen IDs.\n"
    )

    if simulate_error:
        if error_type == "missing":
            prompt += (
                "\nDeliberately omit some crucial information (e.g., H. pylori status, margin status, or EGGIM score). "
                "Do not say that anything is missing."
            )
        elif error_type == "wrong":
            prompt += (
                "\nIntroduce one or two subtle clinical inaccuracies (minor inconsistencies in measurements or staging) without stating they are inaccurate."
            )

    print(prompt)
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=MAX_REPORT_TOKENS,
        temperature=0.3
    )
    # 保存prompt
    if patient_name and report_type:
        prompt_filename = os.path.join(REPORTS_DIR, f"{patient_name}_{report_type}_prompt_{'correct' if not simulate_error else 'wrong'}.txt")
        with open(prompt_filename, "w", encoding="utf-8") as f:
            f.write(prompt)
    return resp.choices[0].message.content.strip()


def generate_explanation_for_wrong_report(report_text: str, report_type: str) -> str:
    """Generate a bullet-point list of errors in the wrong report."""
    prompt = (
        "You are auditing a clinical document for internal dataset QA.\n"
        f"Document type: {report_type}.\n\n"
        "==== DOCUMENT START ====\n"
        f"{report_text}\n"
        "==== DOCUMENT END ====\n\n"
        "List the concrete errors, omissions, contradictions, or implausible details you detect. "
        "Be concise and specific. Use short bullet points. Do NOT restate the full document."
    )
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=MAX_EXPLANATION_TOKENS,
        temperature=0.2
    )
    return resp.choices[0].message.content.strip()

def build_extraction_prompt(pathology: str, endoscopy: str) -> str:
    """Build the prompt for structured extraction of 39 fields"""
    lines = [
        "You are a precise information extraction system for gastric reports.",
        "Consider the following TWO documents for the SAME patient:",
        "",
        "=== PATHOLOGY REPORT ===",
        pathology,
        "",
        "=== ENDOSCOPY REPORT ===",
        endoscopy,
        "",
        "Extract a unified set of fields by combining evidence from BOTH reports.",
        "Rules:",
        "- If conflicting info between two reports, choose the most severe; default to 'not_mentioned' if unclear.",
        "- Use EXACT spellings for keys and allowed values (case-sensitive).",
        "- If a value is not present in either report, use 'not_mentioned' (where allowed).",
        "- For numerics: Use 'not_mentioned' if absent; emit as strings in specified formats.",
        "- Extract the treatment recommended or performed for the current neoplastic lesion. Do not consider routine surveillance or follow-up as treatment."
        "- Return STRICTLY a single JSON object with EXACTLY these keys and string values. No extra keys, no arrays, no nesting, no explanations.",
        "",
        "Keys and allowed values (use EXACTLY):",
    ]
    for key in EXTRACTION_KEYS:
        if key in ALLOWED:
            allowed = sorted(list(ALLOWED[key]))
            lines.append(f'{key}: Allowed: {allowed}')
        elif key in NUMERIC_STRING_KEYS:
            if key == "eggim_score":
                lines.append(f'{key}: Allowed: Numeric string (0-10) OR "not_mentioned". Example: "5"')
            elif key == "neoplastic_lesion_size":
                lines.append(f'{key}: Allowed: Numeric with "mm" suffix OR "not_mentioned". Example: "15 mm"')
            elif key.startswith("pepsinogen"):
                lines.append(f'{key}: Allowed: Numeric with "ng/ml" suffix OR "not_mentioned". Example: "42.5 ng/ml"')
            elif key == "patient_age":
                lines.append(f'{key}: Allowed: Numeric age OR "not_mentioned". Example: "65"')
    lines.append("")
    lines.append("Output only the JSON, no prose.")
    return "\n".join(lines)


def extract_structured_from_reports(pathology: str, endoscopy: str) -> Dict[str, Any]:
    """Extract 39 fields from pathology and endoscopy reports."""
    prompt = build_extraction_prompt(pathology, endoscopy)
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=MAX_EXTRACTION_TOKENS,
        temperature=0.0,   # 提取JSON要求每次输出字段和格式绝对一致，不要任何随机性
        top_p=1.0
    )
    raw = resp.choices[0].message.content.strip()

    # Clean markdown code fences
    raw_clean = raw
    if raw_clean.startswith("```"):
        raw_clean = re.sub(r"^```(?:json)?\s*", "", raw_clean, flags=re.IGNORECASE)
        raw_clean = re.sub(r"\s*```$", "", raw_clean)
    try:
        data = json.loads(raw_clean)
    except Exception:
        data = {k: "not_mentioned" for k in EXTRACTION_KEYS}
    return sanitize_extraction(data)


def _coerce_numeric_string(value: str, pattern: str, not_mentioned_label: str) -> str:
    if not isinstance(value, str):
        return not_mentioned_label
    if value.lower() == "not_mentioned":
        return not_mentioned_label
    if re.match(pattern, value.strip(), flags=re.IGNORECASE):
        return value.strip()
    return not_mentioned_label


def sanitize_extraction(data: Dict[str, Any]) -> Dict[str, str]:
    sanitized: Dict[str, str] = {}
    for key in EXTRACTION_KEYS:
        val = data.get(key, None)

        if key in NUMERIC_STRING_KEYS:
            pattern = NUMERIC_STRING_KEYS[key]
            nm = "not_mentioned"
            if isinstance(val, str) and val.lower() == "not_mentioned":
                sanitized[key] = nm
            else:
                sanitized[key] = _coerce_numeric_string(str(val) if val is not None else "", pattern, nm)
            continue

        allowed = ALLOWED.get(key, None)
        if allowed is not None:
            if isinstance(val, str):
                val_lower = val.lower()
                if val_lower in allowed:
                    sanitized[key] = val_lower
                else:
                    sanitized[key] = "not_mentioned" if "not_mentioned" in allowed else sorted(list(allowed))[0]
            else:
                sanitized[key] = "not_mentioned" if "not_mentioned" in allowed else sorted(list(allowed))[0]
        else:
            sanitized[key] = "not_mentioned"
    return sanitized


# =======================
# File I/O Helpers
# =======================

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def build_filename(patient_name: str, report_type: str, suffix: str) -> str:
    safe_patient = "".join(c for c in patient_name if c.isalnum() or c in ("_", "-"))
    safe_type = "".join(c for c in report_type if c.isalnum() or c in ("_", "-"))
    safe_suffix = "".join(c for c in suffix if c.isalnum() or c in ("_", "-"))
    return os.path.join(REPORTS_DIR, f"{safe_patient}_{safe_type}_{safe_suffix}.txt")


def save_report(patient_name: str, report_type: str, category: str, content: str, suffix: str = "correct") -> str:
    ensure_dir(REPORTS_DIR)
    filename = build_filename(patient_name, report_type, suffix)
    header = f"=== {patient_name} ({category}) - {report_type} Report ({suffix.replace('_', ' ').title()}) ===\n\n"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(header)
        f.write(content)
    print(f"Saved: {filename}")
    return filename


def save_explanation_file(patient_name: str, report_type: str, category: str, explanation_text: str, suffix: str) -> str:
    ensure_dir(REPORTS_DIR)
    safe_patient = "".join(c for c in patient_name if c.isalnum() or c in ("_", "-"))
    safe_type = "".join(c for c in report_type if c.isalnum() or c in ("_", "-"))
    safe_suffix = "".join(c for c in suffix if c.isalnum() or c in ("_", "-"))
    filename = os.path.join(REPORTS_DIR, f"{safe_patient}_{safe_type}_explanation_{safe_suffix}.txt")
    header = f"=== {patient_name} ({category}) - {report_type} Report Explanation ({suffix.replace('_', ' ').title()}) ===\n\n"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(header)
        f.write(explanation_text)
    print(f"Saved explanation: {filename}")
    return filename


def save_master_patient_file(
    patient_name: str,
    category: str,
    correct_pathology: str,
    correct_endoscopy: str,
    wrong_pathology: str,
    wrong_endoscopy: str,
    explanation_files: dict,
    meta: dict,
) -> str:
    ensure_dir(REPORTS_DIR)
    ts = meta.get("timestamp", "")
    filename = os.path.join(REPORTS_DIR, f"{patient_name}_MASTER_{ts}.txt")
    pieces = [
        f"# Master Report for {patient_name} ({category})",
        f"Generated: {meta.get('generated_at', '')}",
        "",
        "## Demographics",
        f"- Gender: {meta.get('gender','')}",
        f"- Nationality: {meta.get('nationality','')}",
        f"- Age: {meta.get('age','')}",
        f"- Race: {meta.get('race','')}",
        f"- Lifestyle: {meta.get('lifestyle','')}",
        f"- Family history: {meta.get('family_history','')}",
        "",
        "## Correct Pathology",
        correct_pathology,
        "",
        "## Correct Endoscopy",
        correct_endoscopy,
        "",
        "## Wrong Pathology",
        wrong_pathology,
        "",
        "## Wrong Endoscopy",
        wrong_endoscopy,
        "",
        "## Explanation Files (separate)",
        f"- Pathology explanation file: {explanation_files.get('pathology','')}",
        f"- Endoscopy explanation file: {explanation_files.get('endoscopy','')}",
        "",
    ]
    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(pieces))
    print(f"Saved master file: {filename}")
    return filename


def save_extraction_json_only_fields(patient_name: str, extraction: Dict[str, Any], suffix: str) -> str:
    ensure_dir(REPORTS_DIR)
    filename = os.path.join(REPORTS_DIR, f"{patient_name}_extracted_{suffix}.json")
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(extraction, f, indent=2, ensure_ascii=False)
    print(f"Saved JSON (39 fields): {filename}")
    return filename


def make_zip_archive(source_dir: str, zip_name: str):
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(source_dir):
            for file in files:
                full_path = os.path.join(root, file)
                arcname = os.path.relpath(full_path, start=source_dir)
                zipf.write(full_path, arcname)
    print(f"Created ZIP archive: {zip_name}")


def delete_folder_contents(path: str):
    for root, dirs, files in os.walk(path, topdown=False):
        for name in files:
            try:
                os.remove(os.path.join(root, name))
            except Exception:
                pass
        for name in dirs:
            try:
                os.rmdir(os.path.join(root, name))
            except Exception:
                pass


# =======================
# Main
# =======================
if __name__ == "__main__":
    # --- 新增：生成时间戳文件夹 ---
    import datetime
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
    # 使用全局变量（需要在函数内声明 global）
    global REPORTS_DIR, ZIP_FILENAME
    REPORTS_DIR = f"reports_{timestamp}"
    ZIP_FILENAME = f"all_reports_{timestamp}.zip"
    
    ensure_dir(REPORTS_DIR)   # 创建文件夹
    # ----- 判断是否启用小批量测试模式 -----
    if BATCH_TEST_CATEGORY.strip():
        # 测试模式：只跑指定类别，生成 full 内容（含 wrong/json）
        target_categories = [BATCH_TEST_CATEGORY]
        num_pairs = BATCH_TEST_NUM_PAIRS
        is_full_output = True
        print(f"BATCH TEST MODE: Running '{BATCH_TEST_CATEGORY}' with {num_pairs} pair(s) — FULL output (wrong + JSON)")
    else:
        # 原有逻辑（Debug 或 Production）
        if DEBUG_MODE:
            target_categories = DEBUG_CATEGORIES
            num_pairs = DEBUG_NUM_PAIRS
            is_full_output = False
            print(f"DEBUG MODE: Running only {target_categories} with {num_pairs} pair(s)")
            print("   (Error reports, explanations, JSON extraction, and ZIP will be skipped)")
        else:
            target_categories = categories
            num_pairs = PRODUCTION_NUM_PAIRS
            is_full_output = True
            print(f"PRODUCTION MODE: Running all categories with {num_pairs} pairs each")

    global_counter = 0
    for category in target_categories:
        code = CATEGORY_CODES.get(category, category[:3].upper())
        if category == "Post-Endoscopic Resection":
            category_counter = 11   # PER 从9开始
        else:
            category_counter = 10   # 其他类别从7开始
        for pair_idx in range(num_pairs):
            patient_name = f"{code}_Patient_{category_counter + pair_idx:03d}"

            # Demographics
            gender = random.choice(genders)
            nationality = random.choice(allowed_nationalities)
            age = random.randint(30, 80)
            family_history = random.choice(family_histories)
            race = random.choice(races)
            lifestyle = random.choice(lifestyle_options)
            # ----- 生成符合临床逻辑的血清胃蛋白酶原值（仅用于内镜报告） -----
            if category in ["Healthy State"]:
                pg_i = random.randint(55, 85)
                pg_ii = random.randint(5, 15)
            elif category in ["Atrophic Gastritis", "Gastric Intestinal Metaplasia"]:
                pg_i = random.randint(20, 45)
                pg_ii = random.randint(12, 22)
            elif category in ["Low-grade Dysplasia", "High-grade Dysplasia", "Early Gastric Carcinoma"]:
                pg_i = random.randint(15, 35)
                pg_ii = random.randint(10, 20)
            else:  # Post-Endoscopic Resection, Special Populations
                pg_i = random.randint(30, 55)
                pg_ii = random.randint(8, 18)

            # ----- [改进] 生成共享的临床事实（确保内镜-病理一致） -----
            clinical_facts = {}
            
            # ---- 1. 为所有类别生成背景萎缩/肠化参数（除非特别指定） ----
            # 初始化默认值（健康状态用）
            atrophy_antrum = "none"
            atrophy_corpus = "none"
            im_antrum = "none"
            im_corpus = "none"
            hp_status = "negative"
            
            if category in ["Atrophic Gastritis", "Gastric Intestinal Metaplasia"]:
                atrophy_antrum = random.choice(["none", "mild", "moderate", "severe"])
                atrophy_corpus = random.choice(["none", "mild", "moderate", "severe"])
                im_antrum = random.choice(["none", "mild", "moderate", "severe"])
                im_corpus = random.choice(["none", "mild", "moderate", "severe"])
                hp_status = random.choice(["positive", "negative"])
            elif category in ["Low-grade Dysplasia", "High-grade Dysplasia", "Early Gastric Carcinoma", "Post-Endoscopic Resection"]:
                # 这些类别也可能有背景异常，随机生成轻度到中度
                atrophy_antrum = random.choice(["none", "mild", "moderate"])
                atrophy_corpus = random.choice(["none", "mild", "moderate"])
                im_antrum = random.choice(["none", "mild", "moderate"])
                im_corpus = random.choice(["none", "mild", "moderate"])
                hp_status = random.choice(["positive", "negative"])
            elif category == "Healthy State":
                # 健康人群：允许轻度萎缩，但不允许肠化
                atrophy_antrum = "none"
                atrophy_corpus = "none"
                im_antrum = "none"
                im_corpus = "none"
                hp_status = "negative"
                family_history = "No family history"
    
                # 随机决定是否有一个部位出现轻度萎缩（50% 概率）
                if random.random() < 0.5:
                    # 只有萎缩选项，不包含肠化
                    choice = random.choice(["antrum_atrophy", "corpus_atrophy"])
                    if choice == "antrum_atrophy":
                        atrophy_antrum = "mild"
                    else:
                        atrophy_corpus = "mild"
    
                # 确保无肠化、Hp阴性、无家族史
                im_antrum = "none"
                im_corpus = "none"
                hp_status = "negative"
                family_history = "No family history"
    
                # 存入 clinical_facts
                clinical_facts["hp_status"] = hp_status
                clinical_facts["family_history"] = family_history
            

            # 将背景参数存入 clinical_facts（所有类别都有）
            clinical_facts["atrophy_antrum"] = atrophy_antrum
            clinical_facts["atrophy_corpus"] = atrophy_corpus
            clinical_facts["im_antrum"] = im_antrum
            clinical_facts["im_corpus"] = im_corpus
            clinical_facts["hp_status"] = hp_status
            
            # ---- 修正萎缩-肠化逻辑一致性（基于化生性萎缩定义） ----
            # 重度肠化至少对应中度萎缩
            if im_antrum == "severe" and atrophy_antrum in ["none", "mild"]:
                atrophy_antrum = random.choice(["moderate", "severe"])
            if im_corpus == "severe" and atrophy_corpus in ["none", "mild"]:
                atrophy_corpus = random.choice(["moderate", "severe"])

            # 重新存入修正后的值
            clinical_facts["atrophy_antrum"] = atrophy_antrum
            clinical_facts["atrophy_corpus"] = atrophy_corpus


            
            # ---- 2. LGD/HGD 可见性 ----
            if category in ["Low-grade Dysplasia", "High-grade Dysplasia"]:
                visibility = random.choice(["visible", "nonvisible"])
                clinical_facts["has_visible_lesion"] = True
                clinical_facts["is_visible"] = visibility
                if visibility == "visible":
                    # --- 新增：随机决定是否为 indefinite ---
                    if random.random() < 0.2:  # 20% 的概率生成 indefinite 场景
                        clinical_facts["dysplasia_type"] = "indefinite"
                    else:
                    # 保留原有的低级别/高级别分类
                        clinical_facts["dysplasia_type"] = category.lower()  # "low-grade" 或 "high-grade"
                    clinical_facts["has_targeted_biopsy"] = True
                    clinical_facts["lesion_location"] = random.choice(["antrum", "corpus", "incisura"])
                    clinical_facts["paris_class"] = random.choice(["0_iia", "0_iic", "0_iia_iic"])
                    clinical_facts["lesion_size"] = random.randint(5, 25)   # 5-25mm
                    clinical_facts["has_ulcer"] = random.choice([True, False])
                else:
                    clinical_facts["has_targeted_biopsy"] = False
                    clinical_facts["lesion_location"] = None
                    clinical_facts["paris_class"] = None
                    clinical_facts["lesion_size"] = None
                    clinical_facts["has_ulcer"] = None
            else:
                # 非 LGD/HGD 类别，默认无可见病变（但 EGC 将在后面单独覆盖）
                clinical_facts["has_visible_lesion"] = None
                clinical_facts["is_visible"] = None
                clinical_facts["has_targeted_biopsy"] = False
                clinical_facts["lesion_location"] = None
                clinical_facts["paris_class"] = None
                clinical_facts["lesion_size"] = None
                clinical_facts["has_ulcer"] = None

            # ----Early Gastric Carcinoma 特殊处理（覆盖默认值） ----
            if category == "Early Gastric Carcinoma":
                clinical_facts["has_visible_lesion"] = True
                clinical_facts["is_visible"] = "visible"
                clinical_facts["has_targeted_biopsy"] = True
                clinical_facts["lesion_location"] = random.choice(["antrum", "corpus", "incisura"])
                clinical_facts["paris_class"] = random.choice(["0_iia", "0_iic", "0_iia_iic"])
                clinical_facts["lesion_size"] = random.randint(5, 50)   # 允许更大范围
                clinical_facts["has_ulcer"] = random.choice([True, False])
                # 生成分化状态
                clinical_facts["differentiation_status"] = random.choices(
                    ["differentiated", "undifferentiated"],
                    weights=[0.7, 0.3]   # 70% 分化型，30% 未分化型
                )[0]

                # --- 根据 MAPS III Fig.8 判断是否直接手术 ---
                size = clinical_facts["lesion_size"]
                ulcer = clinical_facts["has_ulcer"]
                diff = clinical_facts["differentiation_status"]

                # 条件 1: 未分化型 + (>20mm 或 溃疡) → 手术
                cond1 = (diff == "undifferentiated") and (size > 20 or ulcer)

                # 条件 2: 分化型 + 有溃疡 + >30mm → 手术
                cond2 = (diff == "differentiated") and ulcer and (size > 30)

                # 最终手术决策
                clinical_facts["is_surgery"] = cond1 or cond2 

            # ---- 处理 PER 所需的位置信息(同EGC) ----
            if category == "Post-Endoscopic Resection":
                clinical_facts["has_visible_lesion"] = True
                clinical_facts["is_visible"] = "visible"
                clinical_facts["has_targeted_biopsy"] = True
                clinical_facts["lesion_location"] = random.choice(["antrum", "corpus", "incisura"])
                clinical_facts["paris_class"] = random.choice(["0_iia", "0_iic", "0_iia_iic"])
                clinical_facts["lesion_size"] = random.randint(5, 40)
                clinical_facts["has_ulcer"] = random.choice([True, False])
            
            # ---- 3. 特殊人群场景 ----
            if category == "Special Populations":
                # 随机选择四种场景之一
                sp_scenario = random.choice([
                    "autoimmune_gastritis",
                    "common_variable_immunodeficiency",
                    "gastric_malt_lymphoma_in_remission",
                    "hereditary_syndrome"
                ])
                clinical_facts["sp_scenario"] = sp_scenario
                
                # 如果是遗传性综合征，随机选择具体类型并存入 clinical_facts
                if sp_scenario == "hereditary_syndrome":
                    clinical_facts["specific_syndrome"] = random.choice(["HDGC", "Lynch syndrome", "FAP"])
                
                # 根据具体场景调整背景参数（保留原有逻辑）
                if sp_scenario == "autoimmune_gastritis":
                    clinical_facts["atrophy_antrum"] = "none"
                    clinical_facts["atrophy_corpus"] = "severe"
                    clinical_facts["im_antrum"] = "none"
                    clinical_facts["im_corpus"] = "moderate"
                    clinical_facts["hp_status"] = "negative"
                elif sp_scenario == "common_variable_immunodeficiency":
                    clinical_facts["atrophy_antrum"] = random.choice(["none", "mild"])
                    clinical_facts["atrophy_corpus"] = random.choice(["none", "mild"])
                    clinical_facts["im_antrum"] = random.choice(["none", "mild"])
                    clinical_facts["im_corpus"] = random.choice(["none", "mild"])
                    clinical_facts["hp_status"] = random.choice(["positive", "negative"])
                elif sp_scenario == "gastric_malt_lymphoma_in_remission":
                    clinical_facts["atrophy_antrum"] = random.choice(["none", "mild"])
                    clinical_facts["atrophy_corpus"] = random.choice(["none", "mild"])
                    clinical_facts["im_antrum"] = random.choice(["none", "mild"])
                    clinical_facts["im_corpus"] = random.choice(["none", "mild"])
                    clinical_facts["hp_status"] = random.choice(["positive", "negative"])
                elif sp_scenario == "hereditary_syndrome":
                    clinical_facts["atrophy_antrum"] = random.choice(["none", "mild", "moderate"])
                    clinical_facts["atrophy_corpus"] = random.choice(["none", "mild", "moderate"])
                    clinical_facts["im_antrum"] = random.choice(["none", "mild", "moderate"])
                    clinical_facts["im_corpus"] = random.choice(["none", "mild", "moderate"])
                    clinical_facts["hp_status"] = random.choice(["positive", "negative"])
            else:
                clinical_facts["sp_scenario"] = None

            # ----- 确定标本类型（按临床阶段分离）-----
            if category == "Early Gastric Carcinoma":
                specimen_types = ["biopsy"]
            elif category == "Post-Endoscopic Resection":
                specimen_types = ["biopsy", "esd"]
            elif category in ["Low-grade Dysplasia", "High-grade Dysplasia"] and clinical_facts.get("is_visible") == "visible":
                specimen_types = ["biopsy"]  # 可见异型增生只做活检，切除后评估由PER处理
            else:
                specimen_types = ["biopsy"]   # Healthy, AG, GIM, Special Populations


            # ----- 所有背景参数（包括特殊人群、EGC等）都已确定，现在计算分期 -----
            clinical_facts["specimen_types"] = specimen_types
            eggim = calculate_eggim(
                clinical_facts["im_antrum"], 
                clinical_facts["im_corpus"]
            )
            clinical_facts.update(eggim)

            stages_kt = calculate_stages_and_kt(
                clinical_facts["atrophy_antrum"],
                clinical_facts["atrophy_corpus"],
                clinical_facts["im_antrum"],
                clinical_facts["im_corpus"]
            )
            clinical_facts.update(stages_kt)
            

            # ---- 4. 决定内镜切除方式 (ESD vs EMR) ----  
            if category in ["Post-Endoscopic Resection", "Early Gastric Carcinoma", "Low-grade Dysplasia", "High-grade Dysplasia"]:
                # 默认 ESD
                procedure_type = "ESD"
                is_emr_eligible = False

                # 检查是否符合 EMR 替代条件（需要先有 paris_class, lesion_size, has_ulcer）
                paris = clinical_facts.get("paris_class", "")
                size = clinical_facts.get("lesion_size", 999)
                is_low_risk = (
                    category == "Low-grade Dysplasia" or
                    (category == "Early Gastric Carcinoma" and not clinical_facts.get("has_ulcer", False))
                )

                if paris == "0_iia" and size <= 10 and is_low_risk:
                    # 随机选择 EMR（概率可调，建议 30%）
                    if random.random() < 0.3:
                        procedure_type = "EMR"
                        is_emr_eligible = True
                    # 否则仍为 ESD

                is_emr_eligible = False
                clinical_facts["procedure_type"] = procedure_type
                clinical_facts["is_emr_eligible"] = is_emr_eligible

                        # ----- Generate correct reports (always) -----
            correct_pathology = generate_report(
                "Pathology", patient_name, category, gender, nationality, age,
                family_history, race, lifestyle,
                pg_i=None, pg_ii=None,
                clinical_facts=clinical_facts
            )
            correct_endoscopy = generate_report(
                "Endoscopy", patient_name, category, gender, nationality, age,
                family_history, race, lifestyle,
                pg_i=pg_i, pg_ii=pg_ii,
                clinical_facts=clinical_facts
            )

            save_report(patient_name, "Pathology", category, correct_pathology, "correct")
            save_report(patient_name, "Endoscopy", category, correct_endoscopy, "correct")

            # ----- 始终提取 correct JSON（无论是否生成错误） -----
            extracted_correct = extract_structured_from_reports(correct_pathology, correct_endoscopy)
            save_extraction_json_only_fields(patient_name, extracted_correct, suffix="correct")

            # ----- 可选生成 wrong 内容（由 GENERATE_WRONG 开关控制） -----
            if GENERATE_WRONG and is_full_output:
                error_type_path = random.choice(["missing", "wrong"])
                error_type_endo = random.choice(["missing", "wrong"])

                wrong_pathology = generate_report(
                    "Pathology", patient_name, category, gender, nationality, age,
                    family_history, race, lifestyle,
                    clinical_facts=clinical_facts,
                    simulate_error=True, error_type=error_type_path
                )
                wrong_endoscopy = generate_report(
                    "Endoscopy", patient_name, category, gender, nationality, age,
                    family_history, race, lifestyle,
                    pg_i=pg_i, pg_ii=pg_ii,
                    clinical_facts=clinical_facts,
                    simulate_error=True, error_type=error_type_endo
                )

                save_report(
                    patient_name, "Pathology", category,
                    wrong_pathology,
                    f"wrong_{error_type_path}"
                )
                save_report(
                    patient_name, "Endoscopy", category,
                    wrong_endoscopy,
                    f"wrong_{error_type_endo}"
                )

                # Explanations for wrong reports
                explanation_path_text = generate_explanation_for_wrong_report(wrong_pathology, "Pathology")
                explanation_endo_text = generate_explanation_for_wrong_report(wrong_endoscopy, "Endoscopy")

                pathology_expl_file = save_explanation_file(
                    patient_name, "Pathology", category, explanation_path_text, f"wrong_{error_type_path}"
                )
                endoscopy_expl_file = save_explanation_file(
                    patient_name, "Endoscopy", category, explanation_endo_text, f"wrong_{error_type_endo}"
                )

                # Master file（包含正确和错误内容）
                if MAKE_MASTER_PER_PATIENT:
                    timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
                    meta = {
                        "generated_at": datetime.datetime.now().isoformat(timespec="seconds"),
                        "timestamp": timestamp,
                        "gender": gender,
                        "nationality": nationality,
                        "age": age,
                        "race": race,
                        "lifestyle": lifestyle,
                        "family_history": family_history,
                    }
                    save_master_patient_file(
                        patient_name, category,
                        correct_pathology, correct_endoscopy,
                        wrong_pathology, wrong_endoscopy,
                        {"pathology": pathology_expl_file, "endoscopy": endoscopy_expl_file},
                        meta
                    )

                # 错误 JSON 提取
                extracted_wrong = extract_structured_from_reports(wrong_pathology, wrong_endoscopy)
                save_extraction_json_only_fields(patient_name, extracted_wrong, suffix="wrong")

       

    # ZIP creation only in production
    if is_full_output:
        if not BATCH_TEST_CATEGORY.strip():
            make_zip_archive(REPORTS_DIR, ZIP_FILENAME)
            if CLEAN_AFTER_ZIP:
                delete_folder_contents(REPORTS_DIR)
                print(f"Cleaned contents of folder: {REPORTS_DIR} (ZIP retained: {ZIP_FILENAME})")
        else:
            print("Test mode: ZIP skipped. Check 'reports/' folder for all files.")

BATCH TEST MODE: Running 'Post-Endoscopic Resection' with 6 pair(s) — FULL output (wrong + JSON)
Generate a complete, realistic, and anonymized Pathology report for a patient with Post-Endoscopic Resection.
Patient: 'PER_Patient_011', Gender: Female, Nationality: Cyprus, Age: 77
Race: Asian
Date: use a realistic recent date.
Use standard medical terminology.
BACKGROUND (based on clinical data): corpus atrophy: mild, antrum intestinal metaplasia: moderate, corpus intestinal metaplasia: moderate, lesion location: antrum.
H. pylori status: positive.
OLGA stage: I, OLGIM stage: III(MUST be exactly as stated here).
PATHOLOGY REPORT
Type: Standard biopsies: antrum x2 (lesser and greater curvature), corpus x2 (lesser and greater curvature).
Additionally, targeted biopsies were obtained from the antrum for lesion characterization.
Findings: Based on the background mucosa,
  - Atrophy: antrum none, corpus mild
  - Intestinal metaplasia: antrum moderate, corpus moderate
  - OLGA stage: I
  - OLG